# Advanced ModelPipeline: Configs, Per-Model Settings, Save/Load
# 高级 ModelPipeline：配置、单模型设置、保存/加载

Scenario: demand planning teams often compare conservative baselines, high-variance tree models, and business-specific lag choices before promoting a champion model.

场景：需求计划团队通常需要比较稳健基线、高方差树模型和业务特定滞后窗口，然后再上线冠军模型。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from PipelineTS.pipeline import ModelPipeline, PipelineConfigs

data = make_retail_demand(n_days=220, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-21].copy(), data.iloc[-21:].copy()

In [ ]:
configs = PipelineConfigs([
    ("random_forest", "rf_short_lag", {
        "init_configs": {"n_estimators": 120, "max_depth": 10, "random_state": 42},
        "fit_configs": {},
        "predict_configs": {},
        "pipeline_configs": {"lags": 7, "scaler": None},
    }),
    ("extra_forest", "extra_standard_scaled", {
        "init_configs": {"n_estimators": 160, "max_depth": 12, "random_state": 42},
        "fit_configs": {},
        "predict_configs": {},
        "pipeline_configs": {"lags": 21, "scaler": StandardScaler()},
    }),
    ("multi_output_model", "multi_output_diff", {
        "init_configs": {},
        "fit_configs": {},
        "predict_configs": {},
        "pipeline_configs": {"differential_n": 1},
    }),
])

In [ ]:
pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    configs=configs,
    include_init_config_model=False,
    quantile=0.9,
    cv=2,
    time_limit=90,
)
leaderboard = pipe.fit(train, valid_data=valid)
leaderboard

In [ ]:
best_name = pipe.leader_board_.iloc[0]["model"]
best_model = pipe.get_model(best_name)
best_configs = pipe.get_model_all_configs(best_name)

print("Best model:", best_name)
print("Config keys:", sorted(best_configs.keys()))
print("Failed models:", pipe.failed_models)
print("Skipped models:", pipe.skipped_models)

In [ ]:
pred_best = pipe.predict(21)
pred_named = pipe.predict(21, model_name=best_name)
q_pred = pipe.predict_quantiles(21, levels=[0.5, 0.8, 0.9])

display(pred_best.head())
display(q_pred.head())

In [ ]:
model_path = Path("../tmp_retail_pipeline.pts")
pipe.save(model_path, metadata={"scenario": "retail_demand", "owner": "planning"})

loaded = ModelPipeline.load(model_path)
loaded.predict(7).head()

In [ ]:
pipe.plot(n=21, history_tail=90, lang="zh")
pipe.plot_leaderboard(lang="zh")